In [ ]:
# PyTorch functions/methods helpers

# 8.3.2
def nin_block(in_channels, out_channels, kernel_size, stride=1, padding=0):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )

# 8.3.3
nin_small = nn.Sequential(
    nin_block(1, 8, kernel_size=5, padding=2),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nin_block(8, 16, kernel_size=3, padding=1),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(16, 10, kernel_size=1),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
)

* NiN changes how we think about convolutional blocks.
* Instead of applying one linear filter to each local patch, a **NiN block applies a small network at each location, using 1 by 1 convolutions to mix channels and add nonlinear processing**.

# How to use this notebook

* Run the notebook from top to bottom in a clean kernel.

* The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs.

* Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

# You are done when you can

- explain why a 1 by 1 convolution is a per-location linear layer
- build a NiN block from spatial convolution and 1 by 1 convolutions
- explain why global average pooling can replace a large dense head
- trace logits from channels through global average pooling
- debug a class-count mismatch at the classifier output

In [1]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

# 8.3.0 The Problem This Notebook Solves

Chapter 7.4 showed that a 1 by 1 convolution mixes channels at each spatial location. NiN turns that idea into an architecture pattern:

```text
spatial convolution finds local evidence
1 by 1 convolution mixes feature channels at each location
another 1 by 1 convolution adds more local nonlinear processing
```

"Network in Network" points to the idea that each local patch is processed by a small neural network rather than a single linear filter.

# 8.3.1 A 1 by 1 Convolution Is a Linear Layer Shared Across Locations

At one pixel location, the input is a channel vector.

A 1 by 1 convolution applies the same linear transformation to that vector at every row and column.

The cell proves this by comparing `nn.Conv2d(..., kernel_size=1)` with `F.linear` applied to every pixel's channel vector.

In [2]:
conv1x1 = nn.Conv2d(3, 4, kernel_size=1, bias=True)
X = torch.randn(2, 3, 5, 5)

Y_conv = conv1x1(X) # output size of = 5 - 1 + 1 = 5 for shape of (batch, out_channels, height, width) or (2, 4, 5, 5)
X_pixels = X.permute(0, 2, 3, 1)
Y_linear_pixels = F.linear(X_pixels, conv1x1.weight.squeeze(-1).squeeze(-1), conv1x1.bias) # X shape of (2, 5, 5, 3) @ conv1x1.weight.T of (3, 4) = shape of (2, 5, 5, 4)
                                                                                           # conv1x1.weight went from (4, 3, 1, 1) -> squeeze (-1) to (4, 3, 1) -> squeeze(-1) again to (4, 3)
                                                                                           # Can also write conv1x1.weight.squeeze() to remove all the 1 dimensions at once

Y_linear = Y_linear_pixels.permute(0, 3, 1, 2) # Shape of (2, 5, 5, 4) rearranged into (2, 4, 5, 5)

print("conv output:", shape(Y_conv))
print("linear rebuilt output:", shape(Y_linear))

assert torch.allclose(Y_conv, Y_linear, atol=1e-6)

conv output: (2, 4, 5, 5)
linear rebuilt output: (2, 4, 5, 5)


# 8.3.2 A NiN Block Adds Local Nonlinear Channel Mixing

* A NiN block begins with a normal spatial convolution.
* Then it uses two 1 by 1 convolutions.
* ReLU after each convolution makes the local computation nonlinear.

The important contract:

```text
the spatial convolution can change height and width
the 1 by 1 convolutions keep height and width
the output channel count is the block's chosen width
```

Before running the cell, predict:

- Input shape: `(2, 3, 16, 16)`.
- A 5 by 5 convolution with padding 2 should keep 16 by 16.
- The block should output 8 channels.

In [4]:
def nin_block(in_channels, out_channels, kernel_size, stride=1, padding=0):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )

block = nin_block(3, 8, kernel_size=5, padding=2)
Y = block(torch.randn(2, 3, 16, 16))

# Layer 0 = output size of floor((16 + 2*2 - 5) + 1) = 16 for shape of (batch, out_channels, height, width) or (2, 8, 16, 16)
# Layer 1 ReLU
# Layer 2 = output size of floor((16 - 1) + 1) = 16 for shape of (batch, out_channels, height, width) or (2, 8, 16, 16)
# Layer 3 ReLU
# Layer 4 = output size of floor((16 - 1) + 1) = 16 for shape of (batch, out_channels, height, width) or (2, 8, 16, 16)
# Layer 5 ReLU

print("NiN block output:", shape(Y))
assert shape(Y) == (2, 8, 16, 16)

NiN block output: (2, 8, 16, 16)


# 8.3.3 Global Average Pooling Turns Class Channels Into Logits

Classic CNNs often flatten a spatial feature map and feed it to dense layers. NiN uses a different classifier head:

```text
make the final channel count equal the class count
average each class channel over all spatial positions
use the resulting values as logits
```

Global average pooling reduces `(batch, classes, height, width)` to `(batch, classes, 1, 1)`.

After flattening, the result is `(batch, classes)`.

In [5]:
nin_small = nn.Sequential(
    nin_block(1, 8, kernel_size=5, padding=2),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nin_block(8, 16, kernel_size=3, padding=1),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(16, 10, kernel_size=1),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
)

X = torch.randn(4, 1, 32, 32)
rows, logits = trace_module_shapes(nin_small, X)
for row in rows:
    print(row)

# Layer 0 = shape of (batch, out_channels, height, width) or (4, 8, 32, 32), nin_block only updates the out_channels
# Layer 1 = output size of ((32 - 2)/2 + 1) = 16 for shape of (batch, out_channels, height, width) or (4, 8, 16, 16), this particualr MaxPool2d only halves the spatial dimensions
# Layer 2 = shape of (batch, out_channels, height, width) or (4, 16, 16, 16), nin_block only updates the out_channels
# Layer 3 = output size of ((16 - 2)/2 + 1) = 16 for shape of (batch, out_channels, height, width) or (4, 16, 8, 8), this particualr MaxPool2d only halves the spatial dimensions
# Layer 4 = output size of 8 - 1 + 1 = 8 for shape of (batch, out_channels, height, width) or (4, 10, 8, 8), as out_chanels are updated
# Layer 5 = (4, 10, 1, 1) as height and width are now averaged per dimension (across columns)
# Layer 6 = (4, 10)

assert shape(logits) == (4, 10)

('0', 'Sequential', (4, 8, 32, 32))
('1', 'MaxPool2d', (4, 8, 16, 16))
('2', 'Sequential', (4, 16, 16, 16))
('3', 'MaxPool2d', (4, 16, 8, 8))
('4', 'Conv2d', (4, 10, 8, 8))
('5', 'AdaptiveAvgPool2d', (4, 10, 1, 1))
('6', 'Flatten', (4, 10))


# 8.3.4 Global Average Pooling Reduces Dense-Head Parameters

Flattening a large spatial map into a dense layer can create many parameters.

Global average pooling uses a stronger architectural assumption:

```text
for classification, the final evidence for each class can be averaged over location
```

That discards exact final-map position, but it greatly reduces the classifier head.

**"For each feature, I only care about how strongly it appears across the entire image, not the specific spatial location where that feature appears."**

In [7]:
batch, channels, height, width = 4, 10, 8, 8
dense_head_weights = channels * height * width * 10
gap_head_weights = 0

features = torch.randn(batch, channels, height, width)
gap_logits = F.adaptive_avg_pool2d(features, (1, 1)).flatten(1) # Average across the height and width dimensions first -> (4, 10, 1, 1), then flatten the dimensions -> (4, 10)
                                                                # The average values are preserved; only their tensor shape changes

print("dense head weights for 10 logits:", dense_head_weights)
print("GAP head weights when channels already equal classes:", gap_head_weights)
print("GAP logits shape:", shape(gap_logits))

assert shape(gap_logits) == (batch, channels)
assert gap_head_weights < dense_head_weights

dense head weights for 10 logits: 6400
GAP head weights when channels already equal classes: 0
GAP logits shape: (4, 10)


# 8.3.5 Break It Deliberately: Final Channel Count Is Class Count

With the NiN classifier pattern, the final channel count becomes the number of logits.
* If the task has 10 classes but the model outputs 7 channels, the loss cannot interpret a target label such as 9.
* This failure is a classifier-head contract error, not a convolution error.

In [8]:
bad_logits = torch.randn(3, 7)
targets = torch.tensor([0, 3, 9])
loss_fn = nn.CrossEntropyLoss()

try:
    loss_fn(bad_logits, targets) # Target 9 is invalid because there are only 7 classes, whose valid indices are 0 through 6
except IndexError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected class target 9 to be invalid for 7 logits")

IndexError
Target 9 is out of bounds.


# 8.3 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. Why is a 1 by 1 convolution equivalent to a shared linear layer over channel vectors?
> A 1×1 convolution takes the channel vector at each pixel and applies the same linear transformation to it. It is therefore equivalent to a shared linear layer applied independently to every spatial location

2. What does a NiN block add beyond a single spatial convolution?
> A NiN block adds 1×1 convolutions after the spatial convolution, allowing the network to learn additional nonlinear combinations of the channel features at each spatial location

3. How does global average pooling convert class feature maps into logits?
> Converts by averaging the dimensional values of heights and columns, then flatten them

4. What representation information does global average pooling discard?
> Discard the spatial information of the channels (only cares about how strongly, or the activation averages, of said features/channels on the entire image, not the specific appearance in terms of where it is)

5. Why must the final channel count match the number of classes in a NiN-style head?
> The final channels become the class logits after global average pooling, so the number of final channels must equal the number of classes. Each channel provides one class score


---